<a href="https://colab.research.google.com/github/AiHuzaifa/Ml-Algos-from-scratch/blob/main/Random_Forest_code_from_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌲 Random Forest Algorithm: First Principles & Mathematics

A single Decision Tree is prone to **overfitting**—it will perfectly memorize the training data and fail on new data. A Random Forest fixes this using the "Wisdom of the Crowd." It builds hundreds of slightly constrained, mathematically unique trees and averages their predictions.

### 1. The Core Architecture: Bagging
Random Forest is built on an ensemble technique called **Bagging (Bootstrap Aggregating)**.
It combines two distinct phases:
1. **Bootstrap:** Generating randomized datasets for every single tree.
2. **Aggregating:** Tallying the predictions of all trees at the very end via Majority Vote.

### 2. The Two Layers of Randomization
To ensure the crowd of trees is actually diverse (and not just 100 identical clones), the algorithm forces two strict rules during training:

*   **Bootstrapping (Row Randomness):** Every tree receives a training dataset the *exact same size* as the master dataset. However, the rows are drawn **with replacement**. This means some rows are duplicated, and roughly ~37% of the original data is left completely untouched by each tree (acting as an invisible test set).
*   **The Blindfold (Feature/Column Randomness):** Right before a tree attempts to find a split, a random subset of columns is hidden from it. The tree is forced to calculate Information Gain using only the visible columns. This forces the model to explore secondary patterns instead of relying on the single most dominant feature every time.

---

### 3. The Mathematics (The Greedy Search)
Inside every individual tree, the algorithm still relies on the exact same brute-force Greedy Search and mathematical impurity functions used in a standard Decision Tree.

**1. Gini Impurity (Measuring Chaos):**
Calculates the probability of randomly guessing a label incorrectly within a specific node.
$$Gini = 1 - \sum_{i=1}^{c} p_i^2$$

**2. Information Gain (The Update Scoreboard):**
Measures the exact amount of chaos eradicated by a specific cut. The threshold that yields the highest IG is locked in as the winning rule.
$$IG = Gini_{parent} - \left( \frac{N_{left}}{N_{parent}} Gini_{left} + \frac{N_{right}}{N_{parent}} Gini_{right} \right)$$

---

### 4. How the Code Works (Step-by-Step)
1. **Initialize the Forest:** Create an empty list to store the winning rules of `N` trees.
2. **Bootstrap the Data:** Randomly sample $X$ and $y$ with replacement.
3. **Iterate and Split:** Loop through the features, test every unique threshold, and calculate Information Gain.
4. **Save the Rules:** Store the `best_col`, `best_threshold`, and the majority class for the left and right boxes into a dictionary.
5. **Predict (Inference):** To classify a new point, drop it through every saved dictionary in the forest. Count all the resulting predictions, and output the absolute majority.

---

### 💡 Pro-Tips for Real-World Application
*   **You Cannot Overfit by Adding Trees:** Unlike Neural Networks, adding more trees to a Random Forest will never hurt accuracy. It simply smooths out the variance.
*   **The Elbow Method:** Because adding trees costs CPU time, you track the error rate as you add trees (10, 50, 100). The exact moment the error rate stops dropping and forms a flat line, you stop. For most medium datasets, 100 trees is the industry standard starting point.

In [1]:
import numpy as np

# Set a random seed so you get the exact same data every time you run the script
np.random.seed(42)

# Generate 30 rows of data across 4 continuous features
age = np.random.randint(18, 65, size=30)
income = np.random.randint(30000, 120000, size=30)
time_on_site = np.random.randint(1, 25, size=30)       # minutes
prev_purchases = np.random.randint(0, 10, size=30)     # count of past items

# Combine the 4 columns into a single (30, 4) feature matrix X
X = np.column_stack((age, income, time_on_site, prev_purchases))

# Target y: Did they buy a premium subscription? (0 = No, 1 = Yes)
# We embed a hidden logical pattern (income > 70k OR time > 12) so your trees have something real to find
y_true_logic = (income > 70000) | (time_on_site > 12)
y = np.where(y_true_logic, 1, 0)

# Inject a small amount of random noise (flip 4 random labels) to make the data realistic and imperfect
noise_indices = np.random.choice(30, size=4, replace=False)
y[noise_indices] = 1 - y[noise_indices]

print("Dataset X shape:", X.shape)
print("Target y shape:", y.shape)
print("\nFirst 3 rows of X (Age, Income, Time, Purchases):")
print(X[:3])
print("First 3 labels of y:")
print(y[:3])

Dataset X shape: (30, 4)
Target y shape: (30,)

First 3 rows of X (Age, Income, Time, Purchases):
[[    56  55658      4      8]
 [    46 114478     18      0]
 [    32  48431      8      8]]
First 3 labels of y:
[0 0 0]


In [8]:
trees = 100
columns = X.shape[1]

In [19]:
def find_gini(y_data):
  if len(y_data) == 0:
        return 0.0
  val_unique , val_count = np.unique(y_data , return_counts = True)

  total_val = np.sum(val_count)

  probabilities = val_count/total_val

  gini = 1 - np.sum(probabilities**2)
  return gini

In [20]:
def find_information_gain(y_left,y_right,parent_y):
  weighted_left = len(y_left)/len(parent_y)
  weighted_right = len(y_right)/len(parent_y)

  left_gini = find_gini(y_left)
  right_gini = find_gini(y_right)
  parent_gini = find_gini(parent_y)


  ig = parent_gini - ((left_gini*weighted_left) + (right_gini*weighted_right))

  return ig

In [24]:
forest = []

for tree in range(trees):
  best_gain = -1
  best_col = None
  best_threshold = None
  n_samples = len(X)
  random_in = np.random.choice(n_samples , size = n_samples , replace=True)

  boost_x = X[random_in]
  boost_y = y[random_in]
  for col in range(columns):
    thresholds = np.unique(boost_x[:,col])
    for threshold in thresholds:
      element_left = boost_x[:,col] <= threshold
      element_right = boost_x[:,col] > threshold

      left_y = boost_y[element_left]
      right_y = boost_y[element_right]

      if len(left_y) ==0 or len(right_y)==0:
        continue
      information_gain = find_information_gain(left_y,right_y,boost_y)

      if information_gain > best_gain:
        best_gain = information_gain
        best_col = col
        best_threshold = threshold
        best_Tree = tree

        left_classes, left_counts = np.unique(left_y, return_counts=True)
        prediction_left = left_classes[np.argmax(left_counts)]

        right_classes, right_counts = np.unique(right_y, return_counts=True)
        prediction_right = right_classes[np.argmax(right_counts)]



  forest.append({
          'tree_id': tree,
          'best_col': best_col,
          'best_threshold': best_threshold,
          'pred_left': prediction_left,
          'pred_right': prediction_right
      })





In [29]:
def predict_forest(new_data_point, trained_forest):
    votes = []

    # 1. Pass the new point through every tree in the list
    for tree in trained_forest:
        col_to_check = tree['best_col']
        threshold = tree['best_threshold']

        # 2. Apply the tree's specific True/False rule
        if new_data_point[col_to_check] <= threshold:
            votes.append(tree['pred_left'])
        else:
            votes.append(tree['pred_right'])

    # 3. Tally the votes
    unique_classes, vote_counts = np.unique(votes, return_counts=True)
    winning_class = unique_classes[np.argmax(vote_counts)]

    return winning_class, votes

# --- Test it with a new user ---
# Let's say: Age 40, Income 85000, Time 15 min, Purchases 2
new_user = np.array([40, 85000, 15, 2])

final_prediction, all_100_votes = predict_forest(new_user, forest)

print(f"Final Forest Prediction: Class {final_prediction}")
print(f"Total Votes for Class 1: {all_100_votes.count(1)}")
print(f"Total Votes for Class 0: {all_100_votes.count(0)}")

Final Forest Prediction: Class 1
Total Votes for Class 1: 95
Total Votes for Class 0: 5
